# OpenVINO SSD MobileNet: CPU vs GPU Detection Benchmark

This notebook is based on `mobilenetv2_object_detection.py`. It downloads/converts an
Open Model Zoo SSD MobileNet model, runs object detection on both **CPU** and **GPU**
(if available), and compares inference latency and detection results side by side.

Sections:
1. Setup & imports
2. Model download/conversion (OMZ)
3. Pre/post-processing helpers
4. Load an input image
5. Compile & run on CPU
6. Compile & run on GPU (falls back gracefully if no GPU is present)
7. Compare latency + visualize detections side by side


## 1. Setup & imports

In [2]:
import shutil
import subprocess
import time
import importlib
from pathlib import Path
from typing import Any

import numpy as np

try:
    import openvino as ov
except ImportError as exc:
    raise SystemExit(
        "OpenVINO is not installed in this environment. Run: pip install -U openvino openvino-dev"
    ) from exc

print("OpenVINO version:", ov.__version__)


OpenVINO version: 2024.6.0-17404-4c0f47d2335-releases/2024/6


In [3]:
# Check which devices OpenVINO can see on this machine (e.g. ['CPU'] or ['CPU', 'GPU'])
core = ov.Core()
available_devices = core.available_devices
print("Available devices:", available_devices)
for d in available_devices:
    try:
        full_name = core.get_property(d, "FULL_DEVICE_NAME")
        print(f"  {d}: {full_name}")
    except Exception as e:
        print(f"  {d}: (could not query name: {e})")


Available devices: ['CPU', 'GPU']
  CPU: 13th Gen Intel(R) Core(TM) i7-1365U
  GPU: Intel(R) Iris(R) Xe Graphics (iGPU)


## 2. Model download / conversion

Same logic as the script: use `omz_downloader` / `omz_converter` to fetch and convert
an SSD MobileNet model to IR format. Edit `MODEL_NAME_CANDIDATES` / `PRECISION` below if needed.


In [4]:
MODEL_NAME_CANDIDATES = [
    "ssd_mobilenet_v1_fpn_coco",
    "ssd_mobilenet_v1_coco",
    "ssdlite_mobilenet_v2",
]
MODELS_ROOT = Path("models")
PRECISION = "FP16"  # or "FP32"


def run_cmd(args):
    print(f"$ {' '.join(args)}")
    subprocess.run(args, check=True)


def pick_available_model_name(downloader_bin, requested=None):
    result = subprocess.run(
        [downloader_bin, "--print_all"], check=True, capture_output=True, text=True
    )
    available = {line.strip() for line in result.stdout.splitlines() if line.strip()}

    if requested:
        if requested in available:
            return requested
        raise SystemExit(f"Requested model '{requested}' not available in this OMZ package.")

    for candidate in MODEL_NAME_CANDIDATES:
        if candidate in available:
            return candidate

    raise SystemExit(
        "No supported MobileNet SSD model found. Tried: " + ", ".join(MODEL_NAME_CANDIDATES)
    )


def ensure_omz_model(model_name, output_root, precision):
    downloader = shutil.which("omz_downloader")
    converter = shutil.which("omz_converter")
    if not downloader or not converter:
        raise SystemExit(
            "Open Model Zoo tools are not available. Install with: pip install -U openvino-dev"
        )

    output_root.mkdir(parents=True, exist_ok=True)
    chosen_model = pick_available_model_name(downloader, model_name)
    print(f"Using OMZ model: {chosen_model}")

    candidate_paths = [
        output_root / "intel" / chosen_model / precision / f"{chosen_model}.xml",
        output_root / "public" / chosen_model / precision / f"{chosen_model}.xml",
    ]
    for model_xml in candidate_paths:
        if model_xml.exists():
            print(f"Using existing converted model: {model_xml}")
            return model_xml

    run_cmd([downloader, "--name", chosen_model, "--output_dir", str(output_root)])
    run_cmd([
        converter, "--name", chosen_model,
        "--download_dir", str(output_root),
        "--output_dir", str(output_root),
        "--precisions", precision,
    ])

    for model_xml in candidate_paths:
        if model_xml.exists():
            return model_xml

    looked_up = "\n".join(str(p) for p in candidate_paths)
    raise SystemExit(f"Converted model not found. Checked:\n{looked_up}")


In [5]:
model_xml = ensure_omz_model(None, MODELS_ROOT, PRECISION)
print("Model IR:", model_xml)


Using OMZ model: ssd_mobilenet_v1_fpn_coco
Using existing converted model: models/public/ssd_mobilenet_v1_fpn_coco/FP16/ssd_mobilenet_v1_fpn_coco.xml
Model IR: models/public/ssd_mobilenet_v1_fpn_coco/FP16/ssd_mobilenet_v1_fpn_coco.xml


## 3. Pre/post-processing helpers

Same helpers as the script (image loading, resizing, SSD output parsing, drawing).

In [6]:
def load_rgb_with_pillow(image_path: Path) -> np.ndarray:
    from PIL import Image
    image = Image.open(image_path).convert("RGB")
    return np.array(image, dtype=np.uint8)


def resize_rgb_nearest(image: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    src_h, src_w = image.shape[:2]
    if src_h == target_h and src_w == target_w:
        return image
    y_idx = np.linspace(0, src_h - 1, target_h).astype(np.int32)
    x_idx = np.linspace(0, src_w - 1, target_w).astype(np.int32)
    return image[y_idx][:, x_idx]


def preprocess(input_shape, source_image=None, input_dtype=np.float32):
    if len(input_shape) != 4:
        raise SystemExit(f"Unexpected input shape: {input_shape}")
    n, d1, d2, d3 = input_shape
    if n != 1:
        raise SystemExit(f"Only batch size 1 is supported: {input_shape}")

    if d3 == 3:
        if source_image is None:
            frame = np.random.randint(0, 255, size=(d1, d2, 3), dtype=np.uint8)
        else:
            frame = resize_rgb_nearest(source_image, d1, d2)
        data = frame[np.newaxis, ...]
    elif d1 == 3:
        if source_image is None:
            frame = np.random.randint(0, 255, size=(d2, d3, 3), dtype=np.uint8)
        else:
            frame = resize_rgb_nearest(source_image, d2, d3)
        data = np.transpose(frame, (2, 0, 1))[np.newaxis, ...]
    else:
        raise SystemExit(f"Unsupported input layout: {input_shape}")

    return data.astype(input_dtype), frame


def parse_multi_output_ssd(outputs, threshold):
    named = {str(k): v for k, v in outputs.items()}
    boxes = named.get("detection_boxes")
    scores = named.get("detection_scores")
    classes = named.get("detection_classes")

    if boxes is None or scores is None or classes is None:
        if len(outputs) == 1:
            only = next(iter(outputs.values())).reshape(-1, 7)
            detections = []
            for row in only:
                _, label, score, xmin, ymin, xmax, ymax = row
                if score >= threshold:
                    detections.append((int(label), float(score), float(xmin), float(ymin), float(xmax), float(ymax)))
            return detections
        raise SystemExit("Unsupported detection output format.")

    boxes = np.squeeze(boxes)
    scores = np.squeeze(scores)
    classes = np.squeeze(classes)

    detections = []
    count = min(len(scores), len(classes), len(boxes))
    for i in range(count):
        score = float(scores[i])
        if score < threshold:
            continue
        ymin, xmin, ymax, xmax = boxes[i]
        detections.append((int(classes[i]), score, float(xmin), float(ymin), float(xmax), float(ymax)))
    return detections


def draw_detections_with_pillow(rgb_image, detections):
    from PIL import Image, ImageDraw
    image = Image.fromarray(rgb_image)
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for label, score, xmin, ymin, xmax, ymax in detections:
        x1 = int(max(0, xmin) * width)
        y1 = int(max(0, ymin) * height)
        x2 = int(min(1, xmax) * width)
        y2 = int(min(1, ymax) * height)
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        draw.text((x1 + 2, max(0, y1 - 12)), f"id={label} {score:.2f}", fill="red")
    return image


## 4. Load an input image

Set `IMAGE_PATH` to a real image for meaningful detections. If left as `None`, a random
synthetic image is used (fine for timing comparisons, but won't produce real detections).


In [7]:
IMAGE_PATH = "images/1.jpg"  # e.g. Path("sample.jpg")
THRESHOLD = 0.5

source_image = None
if IMAGE_PATH is not None:
    source_image = load_rgb_with_pillow(Path(IMAGE_PATH))
    print("Loaded image:", IMAGE_PATH, source_image.shape)
else:
    print("No IMAGE_PATH set -- using synthetic random input for benchmarking only.")


Loaded image: images/1.jpg (1000, 1500, 3)


## 5 & 6. Compile & benchmark on each device

This helper compiles the model on a given device, runs a warm-up inference, then times
`N_RUNS` inferences to get an average latency. It also returns the parsed detections and
the annotated frame for visual comparison.


In [11]:
N_RUNS = 420  # number of timed inference iterations per device


def benchmark_device(model_xml: Path, device: str, source_image, threshold: float, n_runs: int = N_RUNS):
    model = core.read_model(model=str(model_xml))
    compiled = core.compile_model(model, device)

    input_port = compiled.input(0)
    input_shape = list(input_port.shape)
    input_type_name = str(input_port.element_type).lower()
    input_dtype = np.uint8 if "u8" in input_type_name else np.float32

    input_tensor, prepared_image = preprocess(input_shape, source_image, input_dtype=input_dtype)

    # Warm-up run (excluded from timing -- first inference includes extra setup cost)
    _ = compiled([input_tensor])

    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        outputs = compiled([input_tensor])
        latencies.append(time.perf_counter() - start)

    detections = parse_multi_output_ssd(outputs, threshold)

    return {
        "device": device,
        "latencies_ms": [l * 1000 for l in latencies],
        "mean_ms": float(np.mean(latencies) * 1000),
        "std_ms": float(np.std(latencies) * 1000),
        "fps": 1.0 / float(np.mean(latencies)),
        "detections": detections,
        "prepared_image": prepared_image,
    }


In [ ]:
results = {}

# --- CPU ---
print("Running on CPU...")
results["CPU"] = benchmark_device(model_xml, "CPU", source_image, THRESHOLD)
print(f"CPU: {results['CPU']['mean_ms']:.2f} ms/frame  ({results['CPU']['fps']:.1f} FPS)  "
      f"detections={len(results['CPU']['detections'])}")


In [12]:
# --- GPU (falls back gracefully if no GPU device is present) ---
if "GPU" in available_devices:
    print("Running on GPU...")
    try:
        results["GPU"] = benchmark_device(model_xml, "GPU", source_image, THRESHOLD)
        print(f"GPU: {results['GPU']['mean_ms']:.2f} ms/frame  ({results['GPU']['fps']:.1f} FPS)  "
              f"detections={len(results['GPU']['detections'])}")
    except Exception as e:
        print(f"GPU run failed: {e}")
else:
    print("No GPU device detected by OpenVINO (available_devices =", available_devices, ")")
    print("On Linux this usually means the Intel GPU driver / compute runtime "
          "(intel-opencl-icd / intel-media-va-driver) isn't installed, or you're on non-Intel GPU hardware "
          "that OpenVINO's GPU plugin doesn't target.")


Running on GPU...
GPU run failed: name 'results' is not defined


## 7. Compare results

In [ ]:
print(f"{'Device':<8} {'Mean (ms)':>10} {'Std (ms)':>10} {'FPS':>8} {'Detections':>11}")
for dev, r in results.items():
    print(f"{dev:<8} {r['mean_ms']:>10.2f} {r['std_ms']:>10.2f} {r['fps']:>8.1f} {len(r['detections']):>11}")

if "CPU" in results and "GPU" in results:
    speedup = results["CPU"]["mean_ms"] / results["GPU"]["mean_ms"]
    faster = "GPU" if speedup > 1 else "CPU"
    print(f"\n{faster} is {max(speedup, 1/speedup):.2f}x faster on this model/hardware.")


In [ ]:
import matplotlib.pyplot as plt

devices = list(results.keys())
means = [results[d]["mean_ms"] for d in devices]
stds = [results[d]["std_ms"] for d in devices]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(devices, means, yerr=stds, capsize=6, color=["#4C72B0", "#DD8452"][: len(devices)])
ax.set_ylabel("Mean latency (ms/frame)")
ax.set_title(f"OpenVINO Inference Latency (avg of {N_RUNS} runs)")
for i, v in enumerate(means):
    ax.text(i, v + max(means) * 0.02, f"{v:.1f} ms", ha="center")
plt.tight_layout()
plt.show()


### Side-by-side annotated frames

Only meaningful if a real `IMAGE_PATH` was set above (random synthetic input rarely
produces confident detections).


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
if len(results) == 1:
    axes = [axes]

for ax, (dev, r) in zip(axes, results.items()):
    annotated = draw_detections_with_pillow(r["prepared_image"], r["detections"])
    ax.imshow(annotated)
    ax.set_title(f"{dev}: {r['mean_ms']:.1f} ms, {len(r['detections'])} detections")
    ax.axis("off")

plt.tight_layout()
plt.show()
